# SLA-MoE — Full sweep on Colab

This notebook runs the complete revision-ready experiment sweep for the JBHI
resubmission on a free Colab GPU.

**Before you run anything, edit cell #1:** set `REPO_URL` to your GitHub repo.

**Repo layout assumption:** the repo's *root* contains `run_all_experiments.py`
and `src/`. (If you instead have a `MoE-main/` subdirectory in the repo, set
`SUBDIR = 'MoE-main'` in cell #1.)

**Steps**
1. Runtime → Change runtime type → T4 GPU.
2. Run cell #1 to clone the repo.
3. Run cell #2 to install dependencies.
4. Run cell #3 to download EEGdenoiseNet.
5. Run cell #4a (quick smoke test) — should finish in a few minutes.
6. Run cells #4b–#6 for the full sweep + ablation + BCI downstream.
7. Run cell #7 to zip and download `results.zip`.

In [ ]:
# 0. GPU sanity check
!nvidia-smi

In [ ]:
# 1. Clone repo  --  EDIT THE TWO VARIABLES BELOW
import os, shutil

REPO_URL = 'https://github.com/lifeinpassion/SLA-MOE.git'   # <-- edit if different
SUBDIR   = ''   # set to 'MoE-main' if your repo has that subfolder; '' if root

# If the repo is PRIVATE, use a Personal Access Token:
#   REPO_URL = 'https://<USERNAME>:<TOKEN>@github.com/<USERNAME>/<REPO>.git'
# Generate the token at https://github.com/settings/tokens (classic, scope: repo).
# Easier path: just make the repo public.

REPO_NAME = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
CLONE_DIR = f'/content/{REPO_NAME}'
WORK_DIR  = os.path.join(CLONE_DIR, SUBDIR) if SUBDIR else CLONE_DIR

# If a previous failed clone left an empty dir, remove it.
if os.path.isdir(CLONE_DIR) and not os.path.isfile(os.path.join(CLONE_DIR, '.git', 'HEAD')):
    print(f'Removing stale {CLONE_DIR}')
    shutil.rmtree(CLONE_DIR, ignore_errors=True)

if not os.path.isdir(CLONE_DIR):
    !git clone $REPO_URL $CLONE_DIR

assert os.path.isdir(WORK_DIR), f'WORK_DIR not found: {WORK_DIR}'
assert os.path.isfile(os.path.join(WORK_DIR, 'run_all_experiments.py')), \
    f'run_all_experiments.py not found in {WORK_DIR} -- check SUBDIR setting'

%cd $WORK_DIR
!ls -la

In [ ]:
# 2. Install deps
!pip install -q torch numpy scipy scikit-learn matplotlib seaborn

In [ ]:
# 3. Download EEGdenoiseNet
import os, urllib.request
os.makedirs('data', exist_ok=True)
BASE = 'https://github.com/ncclabsustech/EEGdenoiseNet/raw/master/data/'
for fname in ['EEG_all_epochs.npy', 'EOG_all_epochs.npy', 'EMG_all_epochs.npy']:
    target = f'data/{fname}'
    if not os.path.exists(target):
        print('Downloading', fname)
        urllib.request.urlretrieve(BASE + fname, target)
!ls -la data/

In [ ]:
# 4a. Smoke test (a few minutes)
!python run_all_experiments.py --mode quick --data-path data

In [ ]:
# 4b. Full sweep — combined EEG+EOG+EMG only first (most important table)
!python run_all_experiments.py --mode full --experiment eeg_eog_emg --data-path data --output-dir results

In [ ]:
# 4c. EEG+EOG and EEG+EMG sweeps (run separately if you hit the session timer)
!python run_all_experiments.py --mode full --experiment eeg_eog --data-path data --output-dir results
!python run_all_experiments.py --mode full --experiment eeg_emg --data-path data --output-dir results

In [ ]:
# 5. Ablation
!python -m src.experiments.ablation --data-path data --output-dir results/ablation

In [ ]:
# 6. Downstream BCI-IV-2a (motor imagery)
!pip install -q moabb mne braindecode
!python -m src.downstream.bci_iv_2a --data-path data --output-dir results/downstream_bci_iv_2a

In [ ]:
# 7. Zip + download results
import shutil
shutil.make_archive('/content/results', 'zip', 'results')
from google.colab import files
files.download('/content/results.zip')